### First few cells is just data preparation

My implementation is build on top of nequip and allegro python libraries to simplify data load and preprocessing.
The wigner simbols are from e3nn
Here are the links
https://github.com/mir-group/nequip
https://github.com/mir-group/allegro

In [1]:
# reference example
from nequip.data import dataset_from_config
from nequip.utils import Config
#from nequip.utils.misc import get_default_device_name
#from nequip.utils.config import _GLOBAL_ALL_ASKED_FOR_KEYS
import os
from nequip.model import model_from_config


default_config = dict(
    root="./",
    tensorboard=False,
    wandb=False,
    model_builders=[
        "SimpleIrrepsConfig",
        "EnergyModel",
        "PerSpeciesRescale",
        "StressForceOutput",
        "RescaleEnergyEtc",
    ],
    dataset_statistics_stride=1,
    device='cuda:0',
    default_dtype="float64",
    model_dtype="float32",
    allow_tf32=True,
    verbose="INFO",
    model_debug_mode=False,
    equivariance_test=False,
    grad_anomaly_mode=False,
    gpu_oom_offload=False,
    append=False,
    warn_unused=False,
    _jit_bailout_depth=2,  # avoid 20 iters of pain, see https://github.com/pytorch/pytorch/issues/52286
    # Quote from eelison in PyTorch slack:
    # https://pytorch.slack.com/archives/CDZD1FANA/p1644259272007529?thread_ts=1644064449.039479&cid=CDZD1FANA
    # > Right now the default behavior is to specialize twice on static shapes and then on dynamic shapes.
    # > To reduce warmup time you can do something like setFusionStrartegy({{FusionBehavior::DYNAMIC, 3}})
    # > ... Although we would wouldn't really expect to recompile a dynamic shape fusion in a model,
    # > provided broadcasting patterns remain fixed
    # We default to DYNAMIC alone because the number of edges is always dynamic,
    # even if the number of atoms is fixed:
    _jit_fusion_strategy=[("DYNAMIC", 3)],
    # Due to what appear to be ongoing bugs with nvFuser, we default to NNC (fuser1) for now:
    # TODO: still default to NNC on CPU regardless even if change this for GPU
    # TODO: default for ROCm?
    _jit_fuser="fuser1",
)

os.environ['NEQUIP_NUM_TASKS'] = '16'
# All default_config keys are valid / requested
#_GLOBAL_ALL_ASKED_FOR_KEYS.update(default_config.keys())

In [2]:
config = Config.from_file('./configs/example_ETN_opt.yaml', defaults=default_config)
    

dataset = dataset_from_config(config, prefix="dataset")

validation_dataset = None

dataset[0]

AtomicData(atom_types=[21, 1], cell=[3, 3], edge_cell_shift=[364, 3], edge_index=[2, 364], forces=[21, 3], pbc=[3], pos=[21, 3], total_energy=[1])

In [3]:
config['parity']

'so3'

In [4]:
# Trainer
from nequip.train.trainer import Trainer
from e3nn import o3

trainer = Trainer(model=None, **Config.as_dict(config))

# what is this
# to update wandb data?
config.update(trainer.params)

# = Train/test split =
trainer.set_dataset(dataset, validation_dataset)

# Some hyperparameteres
Nc = 10 # number of chennels for F features from ETN paper
N_rank_spec = 4 # hidden rank of reduction for type radial tensor
config['Nc'] = Nc
config['N_rank_spec'] = N_rank_spec

# ETN parameters
config['d'] = 4 # dimention of the tensor train
config['N_rank_ett'] = [4, 4, 4] # ranks of tensor train



# = Build model =
final_model = model_from_config(
    config=config, initialize=True, dataset=trainer.dataset_train
)

DEBUG:root:* Initialize Output
  ...generate file name results/aspirin/example/log
  ...open log file results/aspirin/example/log
  ...generate file name results/aspirin/example/metrics_epoch.csv
  ...open log file results/aspirin/example/metrics_epoch.csv
  ...generate file name results/aspirin/example/metrics_initialization.csv
  ...open log file results/aspirin/example/metrics_initialization.csv
  ...generate file name results/aspirin/example/metrics_batch_train.csv
  ...open log file results/aspirin/example/metrics_batch_train.csv
  ...generate file name results/aspirin/example/metrics_batch_val.csv
  ...open log file results/aspirin/example/metrics_batch_val.csv
  ...generate file name results/aspirin/example/best_model.pth
  ...generate file name results/aspirin/example/last_model.pth
  ...generate file name results/aspirin/example/trainer.pth
  ...generate file name results/aspirin/example/config.yaml
Torch device: cuda:0
instantiate Loss
...Loss_param = dict(
...   optional_arg

In [5]:
import torch
from torch.nn.functional import one_hot
from nequip.data import AtomicData, AtomicDataDict
from torch.nn.functional import one_hot
from e3nn.nn import FullyConnectedNet
from allegro import with_edge_spin_length
from allegro import _keys
from torch import nn
import math

trainer.model = final_model

# Test configuration stores as dict of parameters
data0 = AtomicData.to_AtomicDataDict(dataset[0])

In [6]:
import torch
from torch.nn.functional import one_hot
from nequip.data import AtomicData, AtomicDataDict
from torch.nn.functional import one_hot
from e3nn.nn import FullyConnectedNet
    
from torch import nn
import math

# forward pass
data_new = final_model(data0)



### Main module

I'll skip the implementation of feature vector F because it heavily relies on nequip code for neighbor lists operations.
If you want you can look into the following files or ask me to add it to the notebook.

Otherwise look into this file

allegro/modules/ETN.py 

and corresponding layers


In [7]:
from typing import Optional
import math
from e3nn.util.codegen import CodeGenMixin
from torch import fx

import torch
from torch_runstats.scatter import scatter

from nequip.data import AtomicDataDict
from nequip.nn import GraphModuleMixin

from allegro import _keys

from torch import nn
from e3nn import o3


class EdgeFeatures_F(nn.Module, GraphModuleMixin):
    def __init__(self,
                 num_types: int,
                 Nc: int, 
                 num_basis: int = 8, 
                 N_rank_spec: int = 4,
                 irreps_in=None,
                 out_field: str = _keys.EDGE_FEATURES_F):
        
        super().__init__()
        self.out_field = out_field
        
        self.irreps_edge_sh = irreps_in[AtomicDataDict.EDGE_ATTRS_KEY]
        
        # set up irreps
        self._init_irreps(
            irreps_in=irreps_in,
            required_irreps_in=[
                AtomicDataDict.EDGE_ATTRS_KEY,
                AtomicDataDict.EDGE_EMBEDDING_KEY,
                _keys.EDGE_TYPE_KEY
            ],
            irreps_out={out_field: o3.Irreps([(Nc, ir) for _, ir in self.irreps_edge_sh])}
        )
        

        
        # parameters of the system
        self.num_types = num_types # number of types
        
        
        # Parameters of the network
        self.Nc = Nc # number of output features
        self.num_basis = num_basis # number of radial basis functions
        self.N_rank_spec = N_rank_spec # encoding of species after one_hot
        
        # tensors for atomic features encoding
        lmax = self.irreps_edge_sh.lmax # maximum spherical harmonic
        
        self._module = EdgeFeatures_FFunction(
            lmax=lmax,
            num_types=self.num_types,
            Nc=self.Nc,
            num_basis=self.num_basis,
            N_rank_spec=self.N_rank_spec,
            irreps_edge_sh=self.irreps_edge_sh,
        )
        

    def forward(self, data: AtomicDataDict.Type) -> AtomicDataDict.Type:
        data[self.out_field] = self._module(data[AtomicDataDict.EDGE_EMBEDDING_KEY],
                                            data[_keys.EDGE_TYPE_KEY],
                                            data[AtomicDataDict.EDGE_ATTRS_KEY])
        
        return data


class EdgeFeatures_FFunction(CodeGenMixin, torch.nn.Module):
    """Module implementing an MLP according to provided options."""

    in_features: int
    out_features: int

    def __init__(
        self,
         lmax: int,
         num_types: int,
         Nc: int, 
         num_basis: int = 8, 
         N_rank_spec: int = 4,
         irreps_edge_sh: o3.Irreps = o3.Irreps("1x0e + 1x1o + 1x2e")
    ):
        super().__init__()

        
        # Code
        params = {}
        graph = fx.Graph()
        tracer = fx.proxy.GraphAppendingTracer(graph)

        def Proxy(n):
            return fx.Proxy(n, tracer=tracer)

        Q = Proxy(graph.placeholder("x"))
        atom_types_embed = Proxy(graph.placeholder("z"))
        Y = Proxy(graph.placeholder("y"))
        norm_from_last: float = 1.0

        base = torch.nn.Module()

        # make weights
        A = torch.empty(lmax + 1, N_rank_spec, num_types**2)
        A.normal_()
        
        B =  torch.empty(lmax + 1, Nc, num_basis, N_rank_spec)           
        B.normal_()

        # generate code
        params[f"A"] = A
        A = Proxy(graph.get_attr(f"A"))

        params[f"B"] = B
        B = Proxy(graph.get_attr(f"B"))


        # Algo from ETN paper to gen F (notation preserved)
        a = A[:, :, atom_types_embed].squeeze(-1)
        b = torch.einsum('Lrnk,LkE,En->ELr', B, a, Q)
    
        F = torch.concat([torch.einsum('Em,En->Emn', Y[:, slices],
                                       b[:, l]) for l, slices in enumerate(irreps_edge_sh.slices())], dim = -2)

        graph.output(F.node)

        for pname, p in params.items():
            setattr(base, pname, torch.nn.Parameter(p))


        self._codegen_register({"_forward": fx.GraphModule(base, graph)})

    def forward(self, x, z, y):
        return self._forward(x, z, y)

### Usage Example

In [8]:
torch.manual_seed(250)

features_F = EdgeFeatures_F(num_types= config['num_types'],
                            Nc = config['Nc'],
                            N_rank_spec = config['N_rank_spec'],
                            irreps_in = final_model.irreps_out,
                            out_field = _keys.EDGE_FEATURES_F)


In [9]:
data_new_new = features_F(data_new)

In [10]:
data_new_new['atomic_energy'][0]

tensor([-618.1429], grad_fn=<SelectBackward0>)

In [11]:
data_new_new['atomic_energy'][0]

tensor([-618.1429], grad_fn=<SelectBackward0>)

### Some equivariance testing of the final model

In [12]:

import torch
from torch.nn.functional import one_hot
from nequip.data import AtomicData, AtomicDataDict
from torch.nn.functional import one_hot
from e3nn.nn import FullyConnectedNet
from allegro import with_edge_spin_length
from allegro import _keys
from torch import nn
import math

data = data0


import copy

data_rot = {key: torch.clone(data0[key]) for key in data0}

irreps_sh = o3.Irreps('1x0e + 1x1o + 1x2e') #o3.Irreps.spherical_harmonics(lmax=2)
irreps_sh_r = o3.Irreps('1x1o')

alpha, beta, gamma = o3.rand_angles(100)

rot_matrix = irreps_sh.D_from_angles(alpha[0], beta[0], gamma[0])
rot_matrix_r = irreps_sh_r.D_from_angles(alpha[0], beta[0], gamma[0])


data_rot['pos'] = data_rot['pos'] @ rot_matrix_r

In [13]:
torch.manual_seed(32)

data_out = final_model(data)

data_out_rot = final_model(data_rot)


#F = data_out['edge_features_F'] # features
#F_rot =data_out_rot['edge_features_F'] # features from rotated positions
#F_rot_rot = torch.einsum('Njn,jk->Nkn', F_rot, rot_matrix.T) # rotated features from rotated positions

#assert torch.allclose(F, F_rot_rot, atol=1e-05)
#print('F is equivariant')
    
    
ETN_out = data_out['node_features_ETN'] # ETN out features
ETN_out_rot =data_out_rot['node_features_ETN'] # ETN out features from rotated positions
ETN_out_rot_rot = torch.einsum('Njn,jk->Nkn', ETN_out_rot, rot_matrix.T) # rotated ETN out features from rotated positions

assert torch.allclose(ETN_out, ETN_out_rot_rot, atol=1e-05)
print('ETN forward output is equivariant')

at_en = data_out['atomic_energy'] # atomic energy
at_en_rot = data_out_rot['atomic_energy'] # atomic energy from rotated positions

assert torch.allclose(at_en, at_en_rot, atol=1e-05)
print('atomic energy is invariant')

tot_en = data_out['total_energy'] # atomic energy
tot_en_rot = data_out_rot['total_energy'] # atomic energy from rotated positions

assert torch.allclose(tot_en, tot_en_rot, atol=1e-05)
print('total energy is invariant')


f_out = data_out['forces'] # atomic forces
f_out_rot =data_out_rot['forces'] # atomic forces from rotated positions
f_out_rot_rot = f_out_rot @ rot_matrix_r.T # rotated forces from rotated positions

assert torch.allclose(f_out, f_out_rot_rot, atol=1e-05)
print('Output forces are equivariant')

ETN forward output is equivariant
atomic energy is invariant
total energy is invariant
Output forces are equivariant


In [14]:
from typing import List, Optional, Tuple
from math import sqrt

import torch
from torch import fx

from e3nn import o3
from e3nn.util.jit import compile
from e3nn.util import prod
from e3nn.o3 import Instruction

from opt_einsum_fx import jitable, optimize_einsums_full


def ETN_third_order_step_forward(
    base_in1: o3.Irreps,
    mul_in1: int,
    base_in2: o3.Irreps,
    mul_in2: int,
    base_out: o3.Irreps,
    mul_out: int,
    num_paths: int,
) -> Optional[fx.GraphModule]:
    """Returns next feature vector"""
    
    #w3j = (
    #    w3j.to_dense()
    #    .reshape(((num_paths,) if num_paths > 1 else tuple()) + kij_shape)
    #    .contiguous()
    #)

    # Generate the mixer
    w3j_shape = (num_paths,) + (base_out.dim, base_in1.dim, base_in2.dim)
    C_shape = (num_paths,) + (mul_in1, mul_in2, mul_out)

    # generate actual code
    graph_out = fx.Graph()
    tracer = fx.proxy.GraphAppendingTracer(graph_out)

    def Proxy(n):
        return fx.Proxy(n, tracer=tracer)

    # = Function definitions =
    u_in = Proxy(graph_out.placeholder("u_in", torch.Tensor))
    F = Proxy(graph_out.placeholder("F", torch.Tensor))

    w3j = Proxy(graph_out.placeholder("w3j", torch.Tensor))
    w3j = w3j.reshape(w3j_shape)
    
    C = Proxy(graph_out.placeholder("C", torch.Tensor))
    C = C.reshape(C_shape)
    
    
    # convert to strided
    u_in = u_in.reshape(-1, base_in1.dim, C_shape[1])
    F = F.reshape(-1, base_in2.dim, C_shape[2])

    # do the einsum
    
    einstr = f"puvw,zkw,zjv,pkij->ziu"
    u_out = torch.einsum(einstr, C, u_in, F, w3j)
    
    graph_out.output(u_out.node)

    # check graphs
    graph_out.lint()

    # Make GraphModules
    # By putting the constants in a Module rather than a dict,
    # we force FX to copy them as buffers instead of as attributes.
    #
    # FX seems to have resolved this issue for dicts in 1.9, but we support all the way back to 1.8.0.
    constants_root = torch.nn.Module()
    #constants_root.register_buffer("_big_w3j", w3j)
    
    graphmod_out = fx.GraphModule(constants_root, graph_out, class_name="etn_step_forward")

    if True:  # optimize_einsums
        # Note that for our einsums, we can optimize _once_ for _any_ batch dimension
        # and still get the right path for _all_ batch dimensions.
        # This is because our einsums are essentially of the form:
        #    zuvw,ijk,zuvij->zwk    OR     uvw,ijk,zuvij->zwk
        # In the first case, all but one operands have the batch dimension
        #    => The first contraction gains the batch dimension
        #    => All following contractions have batch dimension
        #    => All possible contraction paths have cost that scales linearly in batch size
        #    => The optimal path is the same for all batch sizes
        # For the second case, this logic follows as long as the first contraction is not between the first two operands. Since those two operands do not share any indexes, contracting them first is a rare pathological case. See
        # https://github.com/dgasmith/opt_einsum/issues/158
        # for more details.
        #
        # TODO: consider the impact maximum intermediate result size on this logic
        #         \- this is the `memory_limit` option in opt_einsum
        # TODO: allow user to choose opt_einsum parameters?
        #
        # We use float32 and zeros to save memory and time, since opt_einsum_fx looks only at traced shapes, not values or dtypes.
        batchdim = 4
        example_inputs = (
            torch.zeros((batchdim, base_in1.dim * mul_in1)),
            torch.zeros((batchdim, base_in2.dim * mul_in2)),
            torch.zeros(w3j_shape),
            torch.zeros(
                1,
                prod(C_shape),
            ),
        )
        graphmod_out = jitable(optimize_einsums_full(graphmod_out, example_inputs))

    graphmod_out.C_shape = C_shape
    graphmod_out._dim_in1 = base_in1.dim
    graphmod_out._dim_in2 = base_in2.dim
    graphmod_out._dim_out = base_out.dim
    graphmod_out._mul_out = mul_out
    graphmod_out.C_numel = abs(prod(C_shape))

    return graphmod_out


def Contracter_ETN_ALS(
    base_in1: o3.Irreps,
    mul_in1: int,
    base_in2: o3.Irreps,
    mul_in2: int,
    base_out: o3.Irreps,
    mul_out: int,
    num_paths: int,
):
    
    mod = ETN_third_order_step_forward(
        base_in1 = base_in1,
        mul_in1 = mul_in1,
        base_in2 = base_in2,
        mul_in2 = mul_in2,
        base_out = base_out,
        mul_out = mul_out,
        num_paths = num_paths,
    )

    mod = compile(mod)
    return mod

In [36]:
from typing import Optional, List
import math
import functools

import torch
from torch import nn
from torch_runstats.scatter import scatter

from e3nn import o3
from e3nn.util.jit import compile_mode

from nequip.data import AtomicDataDict
from nequip.nn import GraphModuleMixin
from nequip.utils.tp_utils import tp_path_exists

from allegro import _keys

#from allegro.nn._strided import Contracter_ETN_ALS
from e3nn.o3 import wigner_3j
from torch.nn import Parameter, ParameterList, ModuleList

# Triangular ineguality for path existance
def tri_ineq(l1, l2, l3):
    return max([l1, l2, l3]) <= min([l1 + l2, l2 + l3, l1 + l3])


@compile_mode("script")
class ETN_ALS_A_B_Module_opt(nn.Module, GraphModuleMixin):
    def __init__(self,
                 d: int,
                 N_rank_ett: List[int],
                 Nc: int = 10,
                 num_types: int  = 3,
                 num_basis: int = 8,
                 N_rank_spec: int = 4,
                 avg_num_neighbors: Optional[float] = None,
                 normalize_edge_features_f: bool = True,
                 irreps_in=None,
                 out_field: str = AtomicDataDict.PER_ATOM_ENERGY_KEY
                ):
        
        super().__init__()
        self.out_field = out_field
        
        
        self.d = d
        self.Nc = Nc
        self.register_buffer("N_rank_ett", torch.as_tensor(N_rank_ett, dtype=torch.long))
        
        # set up irreps
        self._init_irreps(
            irreps_in=irreps_in,
            required_irreps_in=[
            ],
            irreps_out={_keys.NODE_FEATURES_ETN: o3.Irreps(
                    [(self.Nc, ir) for _, ir in irreps_in[AtomicDataDict.EDGE_ATTRS_KEY] ]),
                        out_field: o3.Irreps([(1, (0, 1))])}
        )
        
        
        # Parameters of the network
        
        # tensors for atomic features encoding
        lmax = irreps_in[AtomicDataDict.EDGE_ATTRS_KEY].lmax # maximum spherical harmonic
        self.lmax = lmax
        
        # Second order cores(first and last)
        core2_1 = Parameter(torch.empty(lmax+1, 1, self.Nc, N_rank_ett[0]).normal_())
        core2_d = Parameter(torch.empty(lmax+1, N_rank_ett[-1], self.Nc, 1).normal_())


        instructions_1 = [(0, l, l) for l in range(lmax + 1)]
        instructions_d = [(l, l, 0) for l in range(lmax + 1)]
        
        
        # Third order cores
        # Assume irreps does not change 
        base_in1 = o3.Irreps([el[1] for el in irreps_in[AtomicDataDict.EDGE_ATTRS_KEY]])
        base_in2 = o3.Irreps([el[1] for el in irreps_in[AtomicDataDict.EDGE_ATTRS_KEY]])
        base_out = o3.Irreps([el[1] for el in irreps_in[AtomicDataDict.EDGE_ATTRS_KEY]])
        

        # Building instructions
        instructions: List[Tuple[int, int, int]] = []
        tmp_i_out: int = 0
        for i_out, (_, ir_out) in enumerate(base_out):
            for i_1, (_, ir_in1) in enumerate(base_in1):
                for i_2, (_, ir_in2) in enumerate(base_in2):
                    if ir_out in ir_in1 * ir_in2:
                        instructions.append((i_1, i_2, i_out))
        
                        tmp_i_out += 1

                        
        self.instructions = instructions
        
        self.register_buffer("instructions_list_0", torch.as_tensor(instructions_1, dtype = torch.long))
        for i in range(1, d - 1):
            self.register_buffer(f"instructions_list_{i}", torch.as_tensor(instructions, dtype = torch.long))
        self.register_buffer(f"instructions_list_{d - 1}", torch.as_tensor(instructions_d, dtype = torch.long))
        
        # building large w3j
        w3j_values = []
        w3j_index = []
        for ins_i, (i_in1, i_in2, i_out) in enumerate(instructions):
            mul_ir_in1 = base_in1[i_in1]
            mul_ir_in2 = base_in2[i_in2]
            mul_ir_out = base_out[i_out]
    
            assert mul_ir_in1.ir.p * mul_ir_in2.ir.p == mul_ir_out.ir.p
            assert (
                tri_ineq(mul_ir_in1.ir.l, mul_ir_in2.ir.l, mul_ir_out.ir.l)
            )
    
            if mul_ir_in1.dim == 0 or mul_ir_in2.dim == 0 or mul_ir_out.dim == 0:
                raise ValueError
    
            this_w3j = o3.wigner_3j(mul_ir_in1.ir.l, mul_ir_in2.ir.l, mul_ir_out.ir.l)
            this_w3j_index = this_w3j.nonzero()
            w3j_values.append(
                this_w3j[this_w3j_index[:, 0], this_w3j_index[:, 1], this_w3j_index[:, 2]]
            )
            
            this_w3j_index[:, 0] += base_in1[: i_in1].dim
            this_w3j_index[:, 1] += base_in2[: i_in2].dim
            this_w3j_index[:, 2] += base_out[: i_out].dim
            # Now need to flatten the index to be for [pk][ij]
            w3j_index.append(
                torch.cat(
                    (  ins_i  * base_out.dim
                        + this_w3j_index[:, 2].unsqueeze(-1),
                        this_w3j_index[:, 0].unsqueeze(-1) * base_in2.dim
                        + this_w3j_index[:, 1].unsqueeze(-1),
                    ),
                    dim=1,
                )
            )
    
        num_paths: int = len(instructions)
    
        w3j = torch.sparse_coo_tensor(
            indices=torch.cat(w3j_index, dim=0).t(),
            values=torch.cat(w3j_values, dim=0),
            size=(
                num_paths * base_out.dim,
                base_in1.dim * base_in2.dim,
            ),
        ).coalesce()
        
        # in dense, must shape it for einsum:
        kij_shape = (
            base_out.dim,
            base_in1.dim,
            base_in2.dim,
        )
        
        # save to buffer in sparce mode + shape
        self.register_buffer("w3j", w3j)
        self.w3j_shape = (num_paths, ) + kij_shape
        
        # third order free parameters
        self.cores = ParameterList([core2_1] + [Parameter(torch.empty(num_paths, N_rank_ett[r], self.Nc, N_rank_ett[r+1]).normal_()) for r in range(d - 2)] + [core2_d]) 
        
        #self.reset_parameters()


        # Register layers F
        self.edge_F = ModuleList([EdgeFeatures_FFunction(
            lmax=self.lmax,
            num_types=num_types,
            Nc=self.Nc,
            num_basis=num_basis,
            N_rank_spec=N_rank_spec,
            irreps_edge_sh=base_in2,
        )  for r in range(self.d)])

        # To convert to node features
        if normalize_edge_features_f and avg_num_neighbors is not None:
            self._factor = 1.0 / math.sqrt(avg_num_neighbors)
        
        # Register layers tensor
        self.tps = [Contracter_ETN_ALS(base_in1, 
                                   N_rank_ett[r], 
                                   base_in2, 
                                   self.Nc, 
                                   base_out, 
                                   N_rank_ett[r+1], 
                                   num_paths) for r in range(self.d - 2)]

        
        
    def forward(self, data: AtomicDataDict.Type) -> AtomicDataDict.Type:

        edge_center = data[AtomicDataDict.EDGE_INDEX_KEY][0]
        edge_neighbor = data[AtomicDataDict.EDGE_INDEX_KEY][1]
        species = data[AtomicDataDict.ATOM_TYPE_KEY].squeeze(-1)
        
        # Input features
        edge_features_f = self.edge_F[-1](data[AtomicDataDict.EDGE_EMBEDDING_KEY],
                             data[_keys.EDGE_TYPE_KEY],
                             data[AtomicDataDict.EDGE_ATTRS_KEY])
        
        F = scatter(edge_features_f, edge_center, dim=0, dim_size=len(species))
        factor: Optional[float] = self._factor  # torchscript hack for typing
        if factor is not None:
            F = F * factor
        
        # Defining tensors for TorchScript
        u_out = torch.zeros((F.shape[0], F.shape[1], self.N_rank_ett[-1]), dtype=F.dtype,
            device=F.device) # temporary verctor output of etn
            
        
        data[_keys.NODE_FEATURES_ETN] = torch.zeros_like(F, dtype=F.dtype,
            device=F.device) # final feature output
        
        slices = self.irreps_in[AtomicDataDict.EDGE_ATTRS_KEY].slices() # slices over irreps

        # getting w3j in dense mode
        w3j_dense = (
            self.w3j.to_dense()
            .reshape(self.w3j_shape)
            .contiguous()
        )   
        
        #print(F[0, :, 0], F.shape)
        # First transform using second order tensors
        for i, slice in enumerate(slices):
            u_out[:, slice, :] = torch.einsum('ij,Nmj->Nmi', self.cores[-1][i].squeeze(-1), F[:, slice, :])

            
        #print(u_out[0, :, 0])
        # Series third order tensors
        for i in range(self.d - 2, 0, -1):
            # Computing F
            edge_features_f = self.edge_F[i](data[AtomicDataDict.EDGE_EMBEDDING_KEY],
                     data[_keys.EDGE_TYPE_KEY],
                     data[AtomicDataDict.EDGE_ATTRS_KEY])
        
            F = scatter(edge_features_f, edge_center, dim=0, dim_size=len(species))
            factor: Optional[float] = self._factor  # torchscript hack for typing
            if factor is not None:
                F = F * factor

            
            # big contruction
            u_out = self.tps[i-1](u_out, F, w3j_dense, self.cores[i])

        # Last transform using second order tensor
        for i, slice in enumerate(slices):
            data[_keys.NODE_FEATURES_ETN][:, slice, :] = torch.einsum('ij,Nmj->Nmi', self.cores[0][i].squeeze(0), u_out[:, slice, :])
        
        #print(data[_keys.NODE_FEATURES_ETN][0, :, 0])
        # Reduction to scalar
        # Computing F
        edge_features_f = self.edge_F[0](data[AtomicDataDict.EDGE_EMBEDDING_KEY],
                 data[_keys.EDGE_TYPE_KEY],
                 data[AtomicDataDict.EDGE_ATTRS_KEY])
    
        F = scatter(edge_features_f, edge_center, dim=0, dim_size=len(species))
        factor: Optional[float] = self._factor  # torchscript hack for typing
        if factor is not None:
            F = F * factor

        
        data[self.out_field] = (( data[_keys.NODE_FEATURES_ETN] * F ).sum(dim = (-2, -1) )).unsqueeze(-1)
        

        return data

In [37]:
from typing import Optional, List
import math
import functools

import torch
from torch import nn
from torch_runstats.scatter import scatter

from e3nn import o3
from e3nn.util.jit import compile_mode

from nequip.data import AtomicDataDict
from nequip.nn import GraphModuleMixin
from nequip.utils.tp_utils import tp_path_exists

from allegro import _keys
from e3nn.o3 import wigner_3j
from torch.nn import Parameter, ParameterList, ModuleList

# Triangular ineguality for path existance
def tri_ineq(l1, l2, l3):
    return max([l1, l2, l3]) <= min([l1 + l2, l2 + l3, l1 + l3])

def convert_to_dense(cores_old, instruction_list):
    """Converts the path matrix to dense format 
       indexed as l1, :,l2, :, l3, :"""
    
    lmax = cores_old[0].shape[0] - 1
    d = len(cores_old)
    
    cores0 = torch.zeros(lmax + 1, cores_old[0].shape[2], 
                         lmax + 1, cores_old[0].shape[-1])
    
    coresd = torch.zeros(lmax + 1, cores_old[-1].shape[1],
                         lmax + 1, cores_old[-1].shape[2])

    for i in range(lmax + 1):
        cores0[i, :, i, :] =  cores_old[0][i, 0]

        coresd[i, :, i, :] =  cores_old[-1][i, :, :, 0]
    
    cores = [torch.zeros(lmax+1, cores_old[i+1].shape[1], 
                         lmax+1, cores_old[i+1].shape[2],
                         lmax+1, cores_old[i+1].shape[3]) for i in range(d-2)]

    for dd in range(d-2):
        for i, (l1, l2, l3) in enumerate(instruction_list[dd + 1]):
            cores[dd][l1, :, l2, :, l3, :] = cores_old[dd + 1][i]
    
    cores = [cores0] + cores + [coresd] 
    return cores


def convert_to_dense_w3j(w3j, instruction, slices):
    """Converts the path matrix to dense format 
       indexed as l1, :,l2, :, l3, :"""
    
    w3j_dense = {(l1, l2, l3): w3j[i][slices[l3], slices[l1], slices[l2]] for i, (l1, l2, l3) in enumerate(instruction)}
    
    return w3j_dense


@compile_mode("script")
class ETN_ALS_A_B_Module_simple_opt(nn.Module, GraphModuleMixin):
    def __init__(self,
                 d: int,
                 N_rank_ett: List[int],
                 Nc: int = 10,
                 num_types: int  = 3,
                 num_basis: int = 8,
                 N_rank_spec: int = 4,
                 avg_num_neighbors: Optional[float] = None,
                 normalize_edge_features_f: bool = True,
                 irreps_in=None,
                 out_field: str = AtomicDataDict.PER_ATOM_ENERGY_KEY
                ):
        
        super().__init__()
        self.out_field = out_field
        
        
        self.d = d
        self.Nc = Nc
        self.register_buffer("N_rank_ett", torch.as_tensor(N_rank_ett, dtype=torch.long))
        
        # set up irreps
        self._init_irreps(
            irreps_in=irreps_in,
            required_irreps_in=[
            ],
            irreps_out={_keys.NODE_FEATURES_ETN: o3.Irreps(
                    [(self.Nc, ir) for _, ir in irreps_in[AtomicDataDict.EDGE_ATTRS_KEY] ]),
                        out_field: o3.Irreps([(1, (0, 1))])}
        )
        
        
        # Parameters of the network
        
        # tensors for atomic features encoding
        lmax = irreps_in[AtomicDataDict.EDGE_ATTRS_KEY].lmax # maximum spherical harmonic
        self.lmax = lmax
        
        # Second order cores(first and last)
        core2_1 = Parameter(torch.empty(lmax+1, 1, self.Nc, N_rank_ett[0]).normal_())
        core2_d = Parameter(torch.empty(lmax+1, N_rank_ett[-1], self.Nc, 1).normal_())


        instructions_1 = [(0, l, l) for l in range(lmax + 1)]
        instructions_d = [(l, l, 0) for l in range(lmax + 1)]
        
        
        # Third order cores
        # Assume irreps does not change 
        base_in1 = o3.Irreps([el[1] for el in irreps_in[AtomicDataDict.EDGE_ATTRS_KEY]])
        base_in2 = o3.Irreps([el[1] for el in irreps_in[AtomicDataDict.EDGE_ATTRS_KEY]])
        base_out = o3.Irreps([el[1] for el in irreps_in[AtomicDataDict.EDGE_ATTRS_KEY]])
        

        # Building instructions
        instructions: List[Tuple[int, int, int]] = []
        tmp_i_out: int = 0
        for i_out, (_, ir_out) in enumerate(base_out):
            for i_1, (_, ir_in1) in enumerate(base_in1):
                for i_2, (_, ir_in2) in enumerate(base_in2):
                    if ir_out in ir_in1 * ir_in2:
                        instructions.append((i_1, i_2, i_out))
        
                        tmp_i_out += 1

        self.instructions_1 = instructions_1                
        self.instructions = instructions
        self.instructions_d = instructions_d
        
        self.register_buffer("instructions_list_0", torch.as_tensor(instructions_1, dtype = torch.long))
        for i in range(1, d - 1):
            self.register_buffer(f"instructions_list_{i}", torch.as_tensor(instructions, dtype = torch.long))
        self.register_buffer(f"instructions_list_{d - 1}", torch.as_tensor(instructions_d, dtype = torch.long))
        
        # building large w3j
        w3j_values = []
        w3j_index = []
        for ins_i, (i_in1, i_in2, i_out) in enumerate(instructions):
            mul_ir_in1 = base_in1[i_in1]
            mul_ir_in2 = base_in2[i_in2]
            mul_ir_out = base_out[i_out]
    
            assert mul_ir_in1.ir.p * mul_ir_in2.ir.p == mul_ir_out.ir.p
            assert (
                tri_ineq(mul_ir_in1.ir.l, mul_ir_in2.ir.l, mul_ir_out.ir.l)
            )
    
            if mul_ir_in1.dim == 0 or mul_ir_in2.dim == 0 or mul_ir_out.dim == 0:
                raise ValueError
    
            this_w3j = o3.wigner_3j(mul_ir_in1.ir.l, mul_ir_in2.ir.l, mul_ir_out.ir.l)
            this_w3j_index = this_w3j.nonzero()
            w3j_values.append(
                this_w3j[this_w3j_index[:, 0], this_w3j_index[:, 1], this_w3j_index[:, 2]]
            )
    
            
            this_w3j_index[:, 0] += base_in1[: i_in1].dim
            this_w3j_index[:, 1] += base_in2[: i_in2].dim
            this_w3j_index[:, 2] += base_out[: i_out].dim
            # Now need to flatten the index to be for [pk][ij]
            w3j_index.append(
                torch.cat(
                    (  ins_i  * base_out.dim
                        + this_w3j_index[:, 2].unsqueeze(-1),
                        this_w3j_index[:, 0].unsqueeze(-1) * base_in2.dim
                        + this_w3j_index[:, 1].unsqueeze(-1),
                    ),
                    dim=1,
                )
            )
    
        num_paths: int = len(instructions)
    
        w3j = torch.sparse_coo_tensor(
            indices=torch.cat(w3j_index, dim=0).t(),
            values=torch.cat(w3j_values, dim=0),
            size=(
                num_paths * base_out.dim,
                base_in1.dim * base_in2.dim,
            ),
        ).coalesce()
        
        # in dense, must shape it for einsum:
        kij_shape = (
            base_out.dim,
            base_in1.dim,
            base_in2.dim,
        )
        
        # save to buffer in sparce mode + shape
        self.register_buffer("w3j", w3j)
        self.w3j_shape = (num_paths, ) + kij_shape
        
        # third order free parameters
        self.cores = ParameterList([core2_1] + [Parameter(torch.empty(num_paths, N_rank_ett[r], self.Nc, N_rank_ett[r+1]).normal_()) for r in range(d - 2)] + [core2_d]) 
        
        #self.reset_parameters()


        # Register layers F
        self.edge_F = ModuleList([EdgeFeatures_FFunction(
            lmax=self.lmax,
            num_types=num_types,
            Nc=self.Nc,
            num_basis=num_basis,
            N_rank_spec=N_rank_spec,
            irreps_edge_sh=base_in2,
        )  for r in range(self.d)])

        # To convert to node features
        if normalize_edge_features_f and avg_num_neighbors is not None:
            self._factor = 1.0 / math.sqrt(avg_num_neighbors)
        
        # Register layers tensor
        self.tps = [Contracter_ETN_ALS(base_in1, 
                                   N_rank_ett[r], 
                                   base_in2, 
                                   self.Nc, 
                                   base_out, 
                                   N_rank_ett[r+1], 
                                   num_paths) for r in range(self.d - 2)]

        
        
    def forward(self, data: AtomicDataDict.Type) -> AtomicDataDict.Type:

        edge_center = data[AtomicDataDict.EDGE_INDEX_KEY][0]
        edge_neighbor = data[AtomicDataDict.EDGE_INDEX_KEY][1]
        species = data[AtomicDataDict.ATOM_TYPE_KEY].squeeze(-1)
        
        # Input features
        edge_features_f = self.edge_F[-1](data[AtomicDataDict.EDGE_EMBEDDING_KEY],
                             data[_keys.EDGE_TYPE_KEY],
                             data[AtomicDataDict.EDGE_ATTRS_KEY])
        
        F = scatter(edge_features_f, edge_center, dim=0, dim_size=len(species))
        factor: Optional[float] = self._factor  # torchscript hack for typing
        if factor is not None:
            F = F * factor
        
        # Defining tensors for TorchScript
        u_out = torch.zeros((F.shape[0], F.shape[1], self.N_rank_ett[-1]), dtype=F.dtype,
            device=F.device) # temporary verctor output of etn
            
        
        data[_keys.NODE_FEATURES_ETN] = torch.zeros_like(F, dtype=F.dtype,
            device=F.device) # final feature output
        
        slices = self.irreps_in[AtomicDataDict.EDGE_ATTRS_KEY].slices() # slices over irreps
        lmax = len(slices) - 1
        
        # getting w3j in dense mode
        w3j_dense = (
            self.w3j.to_dense()
            .reshape(self.w3j_shape)
            .contiguous()
        )   
        
        w3j_big = convert_to_dense_w3j(w3j_dense, self.instructions, slices)
        
        print(F[0, :, 0], F.shape)
        
        instruction_list = [self.instructions_1] + [self.instructions for _ in range(self.d - 2)] + [self.instructions_d]
        
        cores = convert_to_dense(self.cores, instruction_list)
        
        # First transform using second order tensors
        for i, slice in enumerate(slices):
            #u_out[:, slice, :] = torch.einsum('ij,Nmj->Nmi', self.cores[-1][i].squeeze(-1), F[:, slice, :])
            u_out[:, slice, :] = torch.einsum('ij,Nmj->Nmi', cores[-1][i, :, i, :], F[:, slice, :])
            
        print(u_out[0, :, 0])
        # Series third order tensors
        for i in range(self.d - 2, 0, -1):
            # Computing F
            edge_features_f = self.edge_F[i](data[AtomicDataDict.EDGE_EMBEDDING_KEY],
                     data[_keys.EDGE_TYPE_KEY],
                     data[AtomicDataDict.EDGE_ATTRS_KEY])
        
            F = scatter(edge_features_f, edge_center, dim=0, dim_size=len(species))
            if factor is not None:
                F = F * factor

            
            
            T2_shape = cores[i].shape
            # big contruction
            # TODO: now define localy, mb define for all, to ensure computational graph
            T_2_tmp = [[torch.zeros(F.shape[0], 2*l1+1, T2_shape[1], 2*l2+1, T2_shape[3], dtype=F.dtype, device=F.device) for l2 in range(lmax + 1)] for l1 in range(lmax + 1)] # result of first reduction of order 3 tensor

            #print(i)
            # First contraction with previous feature vector
            for l1 in range(lmax + 1):
                for l2 in range(lmax + 1):
                    for l3, slice in enumerate(slices):
                        if (l1, l2, l3) in w3j_big.keys():
                            #T_3 = self.w3j_big[l1][l2][l3][..., None, None, None] * self.cores3[i][(l1, l2, l3)][None, None, None, ...]
                            T_2_tmp[l1][l2] += torch.einsum('kij,uvw,Nkw->Niujv', w3j_big[(l1, l2, l3)], cores[i][l1, :, l2, :, l3, :], u_out[:, slice, :])


           # Second contraction with F vector
            u_out_new = torch.zeros((F.shape[0], F.shape[1], T2_shape[1]), dtype=F.dtype,
                            device=F.device) # temporary verctor output of etn

            for l1 in range(lmax + 1):    
                for l2, slice in enumerate(slices):
                    u_out_new[:, slices[l1], :] += torch.einsum('Niujv,Njv->Niu', T_2_tmp[l1][l2], F[:, slice, :])


            u_out = u_out_new

        # Last transform using second order tensor
        for i, slice in enumerate(slices):
            data[_keys.NODE_FEATURES_ETN][:, slice, :] = torch.einsum('ij,Nmj->Nmi', cores[0][i, :, i, :], u_out[:, slice, :])
            
        print(data[_keys.NODE_FEATURES_ETN][0, :, 0])
        
        # Reduction to scalar
        # Computing F
        edge_features_f = self.edge_F[0](data[AtomicDataDict.EDGE_EMBEDDING_KEY],
                 data[_keys.EDGE_TYPE_KEY],
                 data[AtomicDataDict.EDGE_ATTRS_KEY])
    
        F = scatter(edge_features_f, edge_center, dim=0, dim_size=len(species))
        if factor is not None:
            F = F * factor

        
        data[self.out_field] = (( data[_keys.NODE_FEATURES_ETN] * F ).sum(dim = (-2, -1) )).unsqueeze(-1)
        

        return data

In [38]:
torch.manual_seed(400)

ETN_ALS_A_B = ETN_ALS_A_B_Module_opt( d = config['d'],
                            N_rank_ett = [4 for _ in range(config['d'] - 1)],
                            num_types= config['num_types'],
                            Nc = config['Nc'],
                            N_rank_spec = config['N_rank_spec'],
                            irreps_in = final_model.irreps_out,
                            out_field = AtomicDataDict.PER_ATOM_ENERGY_KEY,
                            avg_num_neighbors = 15)

In [39]:
print("WITH CONTRACTER")
print()

sample = {key: torch.clone(data_new[key]) for key in data_new}

data_contracter = ETN_ALS_A_B(sample)


print("Zero energy check")
print(data_contracter[AtomicDataDict.PER_ATOM_ENERGY_KEY][0])

print("Total energy check")
print(data_contracter[AtomicDataDict.PER_ATOM_ENERGY_KEY].sum())

WITH CONTRACTER

Zero energy check
tensor([-127582.0312], grad_fn=<SelectBackward0>)
Total energy check
tensor(3660778., grad_fn=<SumBackward0>)


In [40]:
torch.manual_seed(400)

ETN_ALS_A_B_simple = ETN_ALS_A_B_Module_simple_opt( d = config['d'],
                            N_rank_ett = [4 for _ in range(config['d'] - 1)],
                            num_types= config['num_types'],
                            Nc = config['Nc'],
                            N_rank_spec = config['N_rank_spec'],
                            irreps_in = final_model.irreps_out,
                            out_field = AtomicDataDict.PER_ATOM_ENERGY_KEY,
                            avg_num_neighbors = 15)

In [41]:
print("WITH LOOPS")
print()

sample = {key: torch.clone(data_new[key]) for key in data_new}

data_loops = ETN_ALS_A_B(sample)


print("Zero energy check")
print(data_loops[AtomicDataDict.PER_ATOM_ENERGY_KEY][0])

print("Total energy check")
print(data_loops[AtomicDataDict.PER_ATOM_ENERGY_KEY].sum())

WITH LOOPS

Zero energy check
tensor([-127582.0312], grad_fn=<SelectBackward0>)
Total energy check
tensor(3660778., grad_fn=<SumBackward0>)


In [42]:
print("CHECK THE DIFF")
print()
if torch.allclose(data_loops[AtomicDataDict.PER_ATOM_ENERGY_KEY], 
                  data_contracter[AtomicDataDict.PER_ATOM_ENERGY_KEY]):
    print("Atomic energies are the same")
    
if torch.allclose(data_loops['node_features_ETN'], 
                  data_contracter['node_features_ETN']):
    print("ETN output is the same")

    
if torch.allclose(data_loops['forces'], 
                  data_contracter['forces']):
    print("Forces output are the same")


CHECK THE DIFF

Atomic energies are the same
ETN output is the same
Forces output are the same


### Check on dense w3j

In [43]:
w3j_dense = (
            ETN_ALS_A_B_simple.w3j.to_dense()
            .reshape(ETN_ALS_A_B_simple.w3j_shape)
            .contiguous()
        )


base_in1 = o3.Irreps([el[1] for el in ETN_ALS_A_B_simple.irreps_in[AtomicDataDict.EDGE_ATTRS_KEY]])
base_in2 = o3.Irreps([el[1] for el in ETN_ALS_A_B_simple.irreps_in[AtomicDataDict.EDGE_ATTRS_KEY]])
base_out = o3.Irreps([el[1] for el in ETN_ALS_A_B_simple.irreps_in[AtomicDataDict.EDGE_ATTRS_KEY]])


# Building instructions
instruction = ETN_ALS_A_B_simple.instructions[10]

l1, l2, l3 = instruction

slices = base_in1.slices()


my_wigner = torch.stack([el for el in w3j_dense[10][slices[l3], slices[l1], slices[l2]]], axis = -1)

assert(torch.allclose(my_wigner, wigner_3j(*instruction)))

print("Conversion of w3j to dense works as expected")

Conversion of w3j to dense works as expected


### Orthogonality check

In [44]:
from allegro import lr_orthogonal

instructions_list = [ETN_ALS_A_B.instructions_list_0,
                     ETN_ALS_A_B.instructions_list_1,
                     ETN_ALS_A_B.instructions_list_2,
                     ETN_ALS_A_B.instructions_list_3]

d = ETN_ALS_A_B.d
ranks = [1] + ETN_ALS_A_B.N_rank_ett.tolist() + [1]
cores = ETN_ALS_A_B.cores

In [45]:
torch.manual_seed(400)

ETN_ALS_A_B = ETN_ALS_A_B_Module_opt( d = config['d'],
                            N_rank_ett = [4 for _ in range(config['d'] - 1)],
                            num_types= config['num_types'],
                            Nc = config['Nc'],
                            N_rank_spec = config['N_rank_spec'],
                            irreps_in = final_model.irreps_out,
                            out_field = AtomicDataDict.PER_ATOM_ENERGY_KEY,
                            avg_num_neighbors = 15)

sample = {key: torch.clone(data_new[key]) for key in data_new}

data_ref = ETN_ALS_A_B(sample)


print("Total energy")
data_ref[AtomicDataDict.PER_ATOM_ENERGY_KEY].sum()

Total energy


tensor(3660778., grad_fn=<SumBackward0>)

In [46]:
from allegro import lr_orthogonal, rl_orthogonal
from torch.nn import ParameterList
torch.manual_seed(400)

ETN_ALS_A_B = ETN_ALS_A_B_Module_opt( d = config['d'],
                            N_rank_ett = [4 for _ in range(config['d'] - 1)],
                            num_types= config['num_types'],
                            Nc = config['Nc'],
                            N_rank_spec = config['N_rank_spec'],
                            irreps_in = final_model.irreps_out,
                            out_field = AtomicDataDict.PER_ATOM_ENERGY_KEY,
                            avg_num_neighbors = 15)


new_cores, _ = lr_orthogonal(cores, ranks, instructions_list)

ETN_ALS_A_B.cores = ParameterList(new_cores)


sample = {key: torch.clone(data_new[key]) for key in data_new}

data_lr_ortho = ETN_ALS_A_B(sample)

print("Total energy")
data_lr_ortho[AtomicDataDict.PER_ATOM_ENERGY_KEY].sum()

Total energy


tensor(3660778.5000, grad_fn=<SumBackward0>)

In [47]:
def rl_orthogonal(tt_cores, R, instr):
    """
    Orthogonalize the TT-cores right to left.

    Parameters
    ----------
    tt_cores : list of torch tensors.
        The TT-cores as a list.

    Returns
    -------
    tt_cores : list of torch tensors.
        The orthogonal TT-cores as a list.

    """  
    
    d = len(tt_cores)

    rank_next = R[0]
    
    core_now = tt_cores[0]
    cores_new = d*[None]
        
    
    
    cores_new = d*[None]
    cores_new[-1] = tt_cores[-1]+0
    for i in range(d-1,0,-1):
        # Init instr
        lmax = max([el[0] for el in instr[i]])
        ind_left = [[ii for ii, ir in enumerate(instr[i-1]) if ir[-1] == ll] for ll in range(lmax+1)]
        ind_right = [[ii for ii, ir in enumerate(instr[i]) if ir[0] == ll] for ll in range(lmax+1)]
        

        
        core_next = tt_cores[i - 1]
        for l in range(lmax+1):
            mode_shape = [cores_new[i].shape[2]]
            core_now = (tn.stack([cores_new[i][ind, ...] for ind in ind_right[l]], dim = -3).flatten(1)).t()
        
        
            Qmat, Rmat = QR(core_now)
            rnew = Rmat.shape[0]
            
            # update current core
            cores_new_tmp = tn.reshape(Qmat.T, [rnew]+[len(ind_right[l])] + mode_shape + [-1])
            cores_new[i][ind_right[l]] = cores_new_tmp.transpose(0, 1)
            
            R[i] = cores_new[i].shape[1]
            
            
            # and the i-1 one
            mode_shape = [core_next.shape[2]]
    
            core_next_tmp = tn.reshape(core_next[ind_left[l]],[len(ind_left[l])*core_next.shape[1]*core_next.shape[2],-1])
            core_next_tmp = core_next_tmp @ Rmat.T
            
            if l == 0:
                cores_new[i - 1] = tn.zeros([len(instr[i-1])] + [core_next.shape[1]] + mode_shape + [cores_new[i].shape[1]], device = tt_cores[0].device)
            
            cores_new[i - 1][ind_left[l]] = tn.reshape(core_next_tmp, [len(ind_left[l])] + [core_next.shape[1]] + mode_shape + [-1])
        
    return cores_new, R

In [48]:
#from allegro import lr_orthogonal, rl_orthogonal
from torch.nn import ParameterList
import torch as tn
torch.manual_seed(400)

ETN_ALS_A_B = ETN_ALS_A_B_Module_opt( d = config['d'],
                            N_rank_ett = [4 for _ in range(config['d'] - 1)],
                            num_types= config['num_types'],
                            Nc = config['Nc'],
                            N_rank_spec = config['N_rank_spec'],
                            irreps_in = final_model.irreps_out,
                            out_field = AtomicDataDict.PER_ATOM_ENERGY_KEY,
                            avg_num_neighbors = 15)


new_cores, _ = rl_orthogonal(cores, ranks, instructions_list)

ETN_ALS_A_B.cores = ParameterList(new_cores)


sample = {key: torch.clone(data_new[key]) for key in data_new}

data_rl_ortho = ETN_ALS_A_B(sample)

print("Total energy")
data_rl_ortho[AtomicDataDict.PER_ATOM_ENERGY_KEY].sum()

Total energy


tensor(3660776.5000, grad_fn=<SumBackward0>)

In [49]:
import torch as tn

def QR(mat):
    """
    Compute the QR decomposition. Backend can be changed.

    Parameters
    ----------
    mat : tn array
        DESCRIPTION.

    Returns
    -------
    Q : the Q matrix
    R : the R matrix

    """
    Q,R = tn.linalg.qr(mat)
    return Q, R

def lr_ortho_simple(cores):
    """Simple orthogonalization for cores
       works like in the paper"""
    
    d = len(cores)
    lmax = cores[0].shape[0] - 1
    
    # Core 0
    cores_cur_tmp = cores[0]
    cores_next_tmp = cores[1].flatten(2)
    
    for l in range(lmax + 1):
        Q, R = QR(cores_cur_tmp[l, :, l, :])
        cores[0][l, :, l, :] = Q.reshape(cores[0][l, :, l, :].shape)#/(2*l + 1)
        cores[1][l] = (R @ cores_next_tmp[l]).reshape(cores[1][l].shape)#/(2*l + 1)
        
    
    
    # Cores triple
    for dd in range(1, d-2):
        cores_cur_tmp = cores[dd].flatten(0, -3)
        cores_next_tmp = cores[dd+1].flatten(2)

        for l in range(lmax + 1):
            Q, R = QR(cores_cur_tmp[:, l, :])
            cores[dd][..., l, :] = Q.reshape(cores[dd][..., l, :].shape)#/(2*l + 1)
            cores[dd+1][l] = (R @ cores_next_tmp[l]).reshape(cores[dd + 1][l].shape)#/(2*l + 1)
   

    # Core d
    cores_cur_tmp = cores[-2].flatten(0, -3)
    cores_next_tmp = cores[-1]
    
    for l in range(lmax + 1):
        Q, R = QR(cores_cur_tmp[:, l, :])
        cores[-2][..., l, :] = Q.reshape(cores[-2][..., l, :].shape)#/(2*l + 1)
        cores[-1][l, :, l, :] = (R @ cores_next_tmp[l, :, l, :]).reshape(cores[-1][l, :, l, :].shape)#/(2*l + 1)
    
    return cores

def rl_ortho_simple(cores):
    """Simple orthogonalization for cores
       works like in the paper"""
    
    d = len(cores)
    lmax = cores[0].shape[0] - 1
    
    
    # Core d
    cores_cur_tmp = cores[-1]
    cores_next_tmp = cores[-2].flatten(0, -3)
    
    for l in range(lmax + 1):
        Q, R = QR(cores_cur_tmp[l, :, l, :].t())
        cores[-1][l, :, l, :] = (Q.T).reshape(cores[-1][l, :, l, :].shape)
        
        cores[-2][..., l, :] = (cores_next_tmp[..., l, :] @ R.T).reshape(cores[-2][..., l, :].shape)#/(2*l + 1)
        
    
    
    # Cores triple
    for dd in range(d-2, 1, -1):
        cores_cur_tmp = cores[dd].flatten(2)
        cores_next_tmp = cores[dd-1].flatten(0, -3)

        for l in range(lmax + 1):
            Q, R = QR(cores_cur_tmp[l, :, :].t())
            cores[dd][l] = (Q.T).reshape(cores[dd][l].shape)#/(2*l + 1)
            cores[dd-1][..., l, :] = (cores_next_tmp[..., l, :] @ R.T).reshape(cores[dd - 1][..., l, :].shape)#/(2*l + 1)
   


    # Core 0
    cores_cur_tmp = cores[1].flatten(2)
    cores_next_tmp = cores[0]
    
    for l in range(lmax + 1):
        Q, R = QR(cores_cur_tmp[l].t())
        cores[1][l] = (Q.T).reshape(cores[1][l].shape)#/(2*l + 1)
        cores[0][l, :, l, :] = (cores_next_tmp[l, :, l, :] @ R.T).reshape(cores[0][l, :, l, :].shape)#/(2*l + 1)


    return cores

def convert_to_dense(cores_old, instruction_list):
    """Converts the path matrix to dense format 
       indexed as l1, :,l2, :, l3, :"""
    
    lmax = cores_old[0].shape[0] - 1
    d = len(cores_old)
    
    cores0 = torch.zeros(lmax + 1, cores_old[0].shape[2], 
                         lmax + 1, cores_old[0].shape[-1])
    
    coresd = torch.zeros(lmax + 1, cores_old[-1].shape[1],
                         lmax + 1, cores_old[-1].shape[2])

    for i in range(lmax + 1):
        cores0[i, :, i, :] =  cores_old[0][i, 0]

        coresd[i, :, i, :] =  cores_old[-1][i, :, :, 0]
    
    cores = [torch.zeros(lmax+1, cores_old[i+1].shape[1], 
                         lmax+1, cores_old[i+1].shape[2],
                         lmax+1, cores_old[i+1].shape[3]) for i in range(d-2)]

    for dd in range(d-2):
        for i, (l1, l2, l3) in enumerate(instruction_list[dd + 1]):
            cores[dd][l1, :, l2, :, l3, :] = cores_old[dd + 1][i]
    
    cores = [cores0] + cores + [coresd] 
    return cores

def convert_to_sparce(cores_old, instruction_list):
    """Converts the path matrix to dense format 
       indexed as l1, :,l2, :, l3, :"""
    
    lmax = cores_old[0].shape[0] - 1
    d = len(cores_old)
    
    cores0 = torch.zeros(lmax + 1, 
                         1, 
                         cores_old[0].shape[1],
                         cores_old[0].shape[3])
    
    coresd = torch.zeros(lmax + 1,
                         cores_old[-1].shape[1],
                         cores_old[-1].shape[3],
                         1)

    for i in range(lmax + 1):
        cores0[i, 0, :, :] =  cores_old[0][i, :, i, :]

        coresd[i, :, :, 0] =  cores_old[-1][i, :, i, :]
    
    cores = [torch.zeros(len(instruction_list[i+1]), 
                         cores_old[i+1].shape[1], 
                         cores_old[i+1].shape[3],
                         cores_old[i+1].shape[5]) for i in range(d-2)]

    for dd in range(d-2):
        for i, (l1, l2, l3) in enumerate(instruction_list[dd + 1]):
            cores[dd][i] = cores_old[dd + 1][l1, :, l2, :, l3, :]
    
    cores = [cores0] + cores + [coresd] 
    return cores

In [50]:
torch.manual_seed(400)

ETN_ALS_A_B = ETN_ALS_A_B_Module_opt( d = config['d'],
                            N_rank_ett = [4 for _ in range(config['d'] - 1)],
                            num_types= config['num_types'],
                            Nc = config['Nc'],
                            N_rank_spec = config['N_rank_spec'],
                            irreps_in = final_model.irreps_out,
                            out_field = AtomicDataDict.PER_ATOM_ENERGY_KEY,
                            avg_num_neighbors = 15)

from allegro import lr_orthogonal

instructions_list = [ETN_ALS_A_B.instructions_list_0,
                     ETN_ALS_A_B.instructions_list_1,
                     ETN_ALS_A_B.instructions_list_2,
                     ETN_ALS_A_B.instructions_list_3]


d = ETN_ALS_A_B.d
ranks = [1] + ETN_ALS_A_B.N_rank_ett.tolist() + [1]
cores = ETN_ALS_A_B.cores


print("REF")
print()
sample = {key: torch.clone(data_new[key]) for key in data_new}

data_ref = ETN_ALS_A_B(sample)

print('per atom energy')
print(data_ref[AtomicDataDict.PER_ATOM_ENERGY_KEY][0])
print('sum')
print(data_ref[AtomicDataDict.PER_ATOM_ENERGY_KEY].sum())
cores_dense = convert_to_dense(cores, instructions_list)

cores_dense_new = lr_ortho_simple(cores_dense)

cores_sparce_new = convert_to_sparce(cores_dense_new, instructions_list)

ETN_ALS_A_B.cores = ParameterList(cores_sparce_new)


print("LR ORTOGONAL")
print()

sample = {key: torch.clone(data_new[key]) for key in data_new}

data_lr_ortho = ETN_ALS_A_B(sample)

print('per atom energy')
print(data_lr_ortho[AtomicDataDict.PER_ATOM_ENERGY_KEY][0])
print('sum')
print(data_lr_ortho[AtomicDataDict.PER_ATOM_ENERGY_KEY].sum())

torch.manual_seed(400)

ETN_ALS_A_B = ETN_ALS_A_B_Module_opt( d = config['d'],
                            N_rank_ett = [4 for _ in range(config['d'] - 1)],
                            num_types= config['num_types'],
                            Nc = config['Nc'],
                            N_rank_spec = config['N_rank_spec'],
                            irreps_in = final_model.irreps_out,
                            out_field = AtomicDataDict.PER_ATOM_ENERGY_KEY,
                            avg_num_neighbors = 15)

d = ETN_ALS_A_B.d
ranks = [1] + ETN_ALS_A_B.N_rank_ett.tolist() + [1]
cores = ETN_ALS_A_B.cores


cores_dense =convert_to_dense(cores, instructions_list)

cores_dense_new = rl_ortho_simple(cores_dense)

cores_sparce_new = convert_to_sparce(cores_dense_new, instructions_list)

ETN_ALS_A_B.cores = ParameterList(cores_sparce_new)


print("RL ORTOGONAL")
print()

sample = {key: torch.clone(data_new[key]) for key in data_new}

data_rl_ortho = ETN_ALS_A_B(sample)

print('per atom energy')
print(data_rl_ortho[AtomicDataDict.PER_ATOM_ENERGY_KEY][0])
print('sum')
print(data_rl_ortho[AtomicDataDict.PER_ATOM_ENERGY_KEY].sum())

REF

per atom energy
tensor([-127582.0312], grad_fn=<SelectBackward0>)
sum
tensor(3660778., grad_fn=<SumBackward0>)
LR ORTOGONAL

per atom energy
tensor([-127582.], grad_fn=<SelectBackward0>)
sum
tensor(3660778., grad_fn=<SumBackward0>)
RL ORTOGONAL

per atom energy
tensor([-127582.3125], grad_fn=<SelectBackward0>)
sum
tensor(3660777., grad_fn=<SumBackward0>)


### Check per index

In [56]:
from allegro import lr_orthogonal, rl_orthogonal, lr_orthogonal_ind, rl_orthogonal_ind
from torch.nn import ParameterList
import torch as tn
torch.manual_seed(400)

ETN_ALS_A_B = ETN_ALS_A_B_Module_opt( d = config['d'],
                            N_rank_ett = [4 for _ in range(config['d'] - 1)],
                            num_types= config['num_types'],
                            Nc = config['Nc'],
                            N_rank_spec = config['N_rank_spec'],
                            irreps_in = final_model.irreps_out,
                            out_field = AtomicDataDict.PER_ATOM_ENERGY_KEY,
                            avg_num_neighbors = 15)


sample = {key: torch.clone(data_new[key]) for key in data_new}

data_ortho = ETN_ALS_A_B(sample)

print("Total energy")
print(data_ortho[AtomicDataDict.PER_ATOM_ENERGY_KEY].sum())  

for t in range(300):
    # Backward sweeps
    for i in range(config['d']-1, 0, -1):


        # rl orthogonalize
        cores_new, _ = rl_orthogonal_ind(cores, ranks, instructions_list, i)

        cores[i] = cores_new[i]
        cores[i-1] = cores_new[i-1]

        ETN_ALS_A_B.cores = ParameterList(cores)

        sample = {key: torch.clone(data_new[key]) for key in data_new}

        data_ortho = ETN_ALS_A_B(sample)

        print("Total energy")
        print(data_ortho[AtomicDataDict.PER_ATOM_ENERGY_KEY].sum())


    # Forward sweeps
    for i in range(config['d']-1):

        # lr orthogonalize
        cores_new, _ = lr_orthogonal_ind(cores, ranks, instructions_list, i)

        cores[i] = cores_new[i]
        cores[i+1] = cores_new[i+1]

        ETN_ALS_A_B.cores = ParameterList(cores)

        sample = {key: torch.clone(data_new[key]) for key in data_new}

        data_ortho = ETN_ALS_A_B(sample)

        print("Total energy")
        print(data_ortho[AtomicDataDict.PER_ATOM_ENERGY_KEY].sum())    

Total energy
tensor(3660778., grad_fn=<SumBackward0>)
Total energy
tensor(3660774., grad_fn=<SumBackward0>)
Total energy
tensor(3660773., grad_fn=<SumBackward0>)
Total energy
tensor(3660773.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660773.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660773.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660774.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660774.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660775., grad_fn=<SumBackward0>)
Total energy
tensor(3660775.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660775., grad_fn=<SumBackward0>)
Total energy
tensor(3660774.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660774.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660773.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660773.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660773.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660773.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660774.7500, 

Total energy
tensor(3660771.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660771.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660771., grad_fn=<SumBackward0>)
Total energy
tensor(3660771.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660770.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660770.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660770.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660770.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660770., grad_fn=<SumBackward0>)
Total energy
tensor(3660770.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660770.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660770.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660771.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660771., grad_fn=<SumBackward0>)
Total energy
tensor(3660770., grad_fn=<SumBackward0>)
Total energy
tensor(3660770.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660771., grad_fn=<SumBackward0>)
Total energy
tensor(3660770., grad

Total energy
tensor(3660771.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660770.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660770.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660770.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660771.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660770.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660770.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660770.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660769.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660770.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660769.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660770., grad_fn=<SumBackward0>)
Total energy
tensor(3660770., grad_fn=<SumBackward0>)
Total energy
tensor(3660769.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660769.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660769.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660770.5000, grad_fn=<SumBackward0>)
Total energy
tensor(36

Total energy
tensor(3660781.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660782., grad_fn=<SumBackward0>)
Total energy
tensor(3660781.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660781.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660780.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660781.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660781.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660781.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660782.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660781.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660781.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660781.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660781.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660782., grad_fn=<SumBackward0>)
Total energy
tensor(3660782.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660782.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660783., grad_fn=<SumBackward0>)
Total energy
tensor(366078

Total energy
tensor(3660783.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660784.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660784., grad_fn=<SumBackward0>)
Total energy
tensor(3660784., grad_fn=<SumBackward0>)
Total energy
tensor(3660783., grad_fn=<SumBackward0>)
Total energy
tensor(3660783., grad_fn=<SumBackward0>)
Total energy
tensor(3660784., grad_fn=<SumBackward0>)
Total energy
tensor(3660783.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660783., grad_fn=<SumBackward0>)
Total energy
tensor(3660783.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660783.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660783.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660783.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660784., grad_fn=<SumBackward0>)
Total energy
tensor(3660784.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660784.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660784.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660784.7500, grad_fn=

...

Total energy
tensor(3660785.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660784.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660784.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660785., grad_fn=<SumBackward0>)
Total energy
tensor(3660784.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660785.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660785.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660786.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660786.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660786., grad_fn=<SumBackward0>)
Total energy
tensor(3660786.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660786.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660786.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660786.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660785.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660786.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660786.5000, grad_fn=<SumBackward0>)
Total energy
tensor(36

Total energy
tensor(3660784.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660784., grad_fn=<SumBackward0>)
Total energy
tensor(3660784.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660784.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660784.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660784.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660783.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660784., grad_fn=<SumBackward0>)
Total energy
tensor(3660784.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660783.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660784., grad_fn=<SumBackward0>)
Total energy
tensor(3660784., grad_fn=<SumBackward0>)
Total energy
tensor(3660783.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660784., grad_fn=<SumBackward0>)
Total energy
tensor(3660783.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660784., grad_fn=<SumBackward0>)
Total energy
tensor(3660785., grad_fn=<SumBackward0>)
Total energy
tensor(3660785., grad_fn=<Sum

Total energy
tensor(3660782., grad_fn=<SumBackward0>)
Total energy
tensor(3660781.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660782., grad_fn=<SumBackward0>)
Total energy
tensor(3660782., grad_fn=<SumBackward0>)
Total energy
tensor(3660782.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660782.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660782., grad_fn=<SumBackward0>)
Total energy
tensor(3660781.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660780.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660781., grad_fn=<SumBackward0>)
Total energy
tensor(3660781.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660781.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660781.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660781.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660782., grad_fn=<SumBackward0>)
Total energy
tensor(3660781.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660782., grad_fn=<SumBackward0>)
Total energy
tensor(3660782., grad_fn=<Sum

Total energy
tensor(3660782.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660782.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660783.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660784., grad_fn=<SumBackward0>)
Total energy
tensor(3660783., grad_fn=<SumBackward0>)
Total energy
tensor(3660783.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660783.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660783.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660783.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660783.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660785., grad_fn=<SumBackward0>)
Total energy
tensor(3660784., grad_fn=<SumBackward0>)
Total energy
tensor(3660784.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660784.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660785., grad_fn=<SumBackward0>)
Total energy
tensor(3660785.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660785.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660785., grad

tensor(3660790.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660790.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660790., grad_fn=<SumBackward0>)
Total energy
tensor(3660790.5000, grad_fn=<SumBackward0>)
Total energy
tensor(3660790., grad_fn=<SumBackward0>)
Total energy
tensor(3660790., grad_fn=<SumBackward0>)
Total energy
tensor(3660789., grad_fn=<SumBackward0>)
Total energy
tensor(3660789., grad_fn=<SumBackward0>)
Total energy
tensor(3660788., grad_fn=<SumBackward0>)
Total energy
tensor(3660789., grad_fn=<SumBackward0>)
Total energy
tensor(3660790., grad_fn=<SumBackward0>)
Total energy
tensor(3660789., grad_fn=<SumBackward0>)
Total energy
tensor(3660789.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660789.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660788.7500, grad_fn=<SumBackward0>)
Total energy
tensor(3660789.2500, grad_fn=<SumBackward0>)
Total energy
tensor(3660789., grad_fn=<SumBackward0>)
Total energy
tensor(3660788.7500, grad_fn=<SumBackward0>)
Total ene